In [1]:
import h5py
import numpy as np

In [2]:
data_path = "/media/RCPNAS/Data2/sengupta/inversion/mich/data/single_layer.h5"

In [8]:
with h5py.File(data_path, "r") as file:
    bold = np.stack([file[layer]["bold"][:20] for layer in file.keys() if "layer" in layer], axis=1)
    source_location = file["meta"]["source_position"][:20]
    source_layer = file["meta"]["source_layer"][:20]

In [11]:
r = 5  # neighbourhood_radius
T = bold.shape[2]  # should be 3000
t_peak_lo = 350  # ~35s at TR=0.1s
t_peak_hi = 1200  # ~50s

stds = []
for i in range(len(source_location)):
    sh, sw = source_location[i]
    sl = source_layer[i]

    h_lo = max(0, sh - r)
    h_hi = min(bold.shape[3], sh + r + 1)
    w_lo = max(0, sw - r)
    w_hi = min(bold.shape[4], sw + r + 1)

    patch = bold[i, sl, t_peak_lo:t_peak_hi, h_lo:h_hi, w_lo:w_hi]  # [T_peak, H_patch, W_patch]
    stds.append(patch.std())

stds = np.array(stds)
print("per-sample neighbourhood std at peak:", stds)
print("mean:", stds.mean(), "median:", np.median(stds), "min:", stds.min(), "max:", stds.max())

per-sample neighbourhood std at peak: [0.002548  0.002783  0.001953  0.00693   0.001359  0.003183  0.001843
 0.001007  0.0049    0.002237  0.002674  0.0008802 0.0004883 0.001674
 0.00298   0.001268  0.000598  0.0008097 0.        0.000646 ]
mean: 0.002039 median: 0.001759 min: 0.0 max: 0.00693
